In [ ]:
%pip install azure-ai-projects azure-identity openai --quiet

In [ ]:
# This installs all four Azure packages you need for the entire course

%pip install python-dotenv
%pip install azure-ai-inference
%pip install azure-identity  
%pip install azure-ai-projects>= 2.0.0
%pip install openai

In [ ]:
# ---- Cell 2: Load credentials ----

import os                        # os lets us read environment variables
from dotenv import load_dotenv   # load_dotenv reads your .env file

load_dotenv()                    # reads .env and loads all KEY=VALUE pairs into os.environ

# Read your project endpoint from .env
# This is the services.ai.azure.com URL — the Foundry SDK endpoint, NOT the openai.azure.com one
PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]

# Read your model deployment name
MODEL = os.environ["MODEL_DEPLOYMENT_NAME"]

print("PROJECT_ENDPOINT:", PROJECT_ENDPOINT)
print("MODEL:", MODEL)

In [ ]:
# ---- Cell 3: Define your two custom tool functions ----

import json    # json lets us convert Python dicts to strings the agent can read
import random  # random generates fake CV numbers (stands in for real DB calculation)

def get_mbds(region: str) -> str:
    # region: str means this function expects a text input (the region name)
    # -> str means this function will return a text (JSON string)
    """
    Retrieves active MBDs for a given region from the D2O database.
    Use this when the user asks about MBDs in a specific region,
    wants to see what MBDs exist, or needs MBD IDs for further analysis.
    Args:
        region: The region code. Valid values: APAC, EMEA, NA, LATAM, GLOBAL
    Returns:
        A JSON string containing the list of active MBDs for that region.
    """
    # This dict is our fake database — in production this would be a real DB query
    fake_db = {
        "APAC": [
            {"id": "MBD-APAC-0042", "name": "[APAC] Premium Electronics", "category": "CAT-ELEC", "status": "Active"},
            {"id": "MBD-APAC-0087", "name": "[APAC] Food & Beverage Core", "category": "CAT-FOOD", "status": "Active"},
            {"id": "MBD-APAC-0103", "name": "[APAC] Health & Wellness",    "category": "CAT-HLTH", "status": "Active"},
        ],
        "EMEA": [
            {"id": "MBD-EMEA-0011", "name": "[EMEA] Apparel Premium",      "category": "CAT-APPR", "status": "Active"},
            {"id": "MBD-EMEA-0034", "name": "[EMEA] Consumer Electronics", "category": "CAT-ELEC", "status": "Active"},
        ],
        "NA": [
            {"id": "MBD-NA-0021",   "name": "[NA] Automotive Parts",       "category": "CAT-AUTO", "status": "Active"},
            {"id": "MBD-NA-0055",   "name": "[NA] Home & Living Core",      "category": "CAT-HOME", "status": "Active"},
        ],
    }
    # .get(key, default) — looks up the region, returns [] if not found
    result = fake_db.get(region.upper(), [])  # .upper() handles "apac" == "APAC"
    
    if not result:   # if the list is empty, the region wasn't found
        return json.dumps({"error": f"No MBDs found for region: {region}"})
    
    # json.dumps converts a Python dict/list into a JSON string
    # The agent receives this string and reads it as structured data
    return json.dumps({"region": region, "mbds": result, "count": len(result)})


def calculate_cvs(mbd_ids: str) -> str:
    # mbd_ids: str — takes a comma-separated string like "MBD-APAC-0042,MBD-APAC-0087"
    """
    Calculates the Coefficient of Variation (CV) for a list of MBDs.
    Use this when the user wants to calculate CVs, check data stability,
    or run the CV algorithm on specific MBDs.
    Args:
        mbd_ids: Comma-separated list of MBD IDs e.g. "MBD-APAC-0042,MBD-APAC-0087"
    Returns:
        A JSON string with CV results for each requested MBD.
    """
    # Split the comma-separated string into a list, strip whitespace from each ID
    ids = [mid.strip() for mid in mbd_ids.split(",")]
    results = []  # we'll build up results for each MBD
    
    for mbd_id in ids:   # loop through each MBD ID the agent passed in
        # Fake different mean/std_dev per region — in production this hits your DB
        if "APAC" in mbd_id:
            mean, std_dev = 4520000, random.uniform(400000, 900000)
        elif "EMEA" in mbd_id:
            mean, std_dev = 7800000, random.uniform(600000, 1800000)
        else:
            mean, std_dev = 3200000, random.uniform(300000, 800000)
        
        cv = (std_dev / mean) * 100   # THE CV formula: (std dev / mean) × 100
        
        # Classify into D2O tiers using thresholds from CV Methodology doc
        if cv <= 10:
            tier, action = "Tier 1 — Stable",    "None required"
        elif cv <= 20:
            tier, action = "Tier 2 — Acceptable", "Monitor in next review cycle"
        elif cv <= 35:
            tier, action = "Tier 3 — Elevated",   "Investigate root cause within 30 days"
        else:
            tier, action = "Tier 4 — Critical",   "Immediate data quality review required"
        
        results.append({   # append adds one result dict to the results list
            "mbd_id":      mbd_id,
            "mean":        round(mean, 2),      # round() to 2 decimal places
            "std_dev":     round(std_dev, 2),
            "cv_percent":  round(cv, 2),
            "tier":        tier,
            "action":      action
        })
    
    return json.dumps({"cv_results": results})   # return all results as one JSON string

print("✅ Functions defined")

In [ ]:
# ---- Cell 4: Define the tool SCHEMA ----
# This is how you describe your functions TO the agent.
# The agent cannot read your Python code — it only reads this schema.
# Think of this as the "menu entry" for each function.

tools = [
    {
        "type": "function",          # tells the SDK this is a custom Python function tool
        "function": {
            "name": "get_mbds",      # MUST exactly match your Python function name
            "description": (         # plain English — the agent reads this to decide when to call it
                "Retrieves active MBDs for a given region from the D2O database. "
                "Use when the user asks about MBDs in a specific region."
            ),
            "parameters": {
                "type": "object",         # parameters is always an object (dict)
                "properties": {
                    "region": {
                        "type": "string", # the agent must pass a string for this argument
                        "description": "Region code: APAC, EMEA, NA, LATAM, or GLOBAL"
                    }
                },
                "required": ["region"]    # "region" is mandatory — agent cannot skip it
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_cvs",
            "description": (
                "Calculates Coefficient of Variation (CV) for a list of MBDs. "
                "Use when the user wants CV values, data stability checks, or tier classification."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "mbd_ids": {
                        "type": "string",
                        "description": "Comma-separated MBD IDs e.g. 'MBD-APAC-0042,MBD-APAC-0087'"
                    }
                },
                "required": ["mbd_ids"]
            }
        }
    }
]

print("✅ Tool schema defined —", len(tools), "tools")

In [ ]:
# ---- Cell 5a: Check and upgrade SDK version ----
%pip show azure-ai-projects

In [ ]:
%pip install azure-ai-projects --upgrade --quiet

In [ ]:
# ---- Cell 5b: Install the dedicated agents package ----
%pip install azure-ai-agents --quiet

In [ ]:
# ---- Cell 5: Create the AIProjectClient and the agent ----

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Create the client — same as before
client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential()
)

# In azure-ai-projects v2.x, agents live under client.agents.create_agent()
# BUT the v2 SDK uses a different import for the agent definition
# Let's check what's available first
print(dir(client.agents))

In [ ]:
# ---- Cell 5: Create AgentsClient and agent ----

from azure.ai.agents import AgentsClient        # dedicated agents SDK — separate from AIProjectClient in v2.x
from azure.identity import DefaultAzureCredential

# AgentsClient is the correct client for agent operations in v2.x
# In v1.x this was inside AIProjectClient — Microsoft split it out in v2
client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,              # same services.ai.azure.com URL
    credential=DefaultAzureCredential()     # same keyless auth
)

agent = client.create_agent(
    model=MODEL,                            # "gpt-4.1" from your .env
    name="ab-custom-tools-agent",           # visible in Foundry portal
    instructions=(
        "You are the D2O Data Assistant. "
        "Always use get_mbds when a region is mentioned. "
        "Always use calculate_cvs when CV calculations are requested. "
        "Present results with tier classification and recommended actions."
    ),
    tools=tools                             # the schema list from Cell 4
)

print("✅ Agent created")
print("Agent ID:", agent.id)
print("Agent name:", agent.name)

In [ ]:
# ---- Cell 6: Create thread and send first message ----

# A Thread = one conversation. We create it once per session.
# client.threads.create() tells Foundry Agent Service: 
# "start a new conversation, give it an ID, store it server-side"
thread = client.threads.create()
print("Thread ID:", thread.id)   # Foundry assigned this thread a unique ID

# Now add the user's first message to the thread
# This is like the user typing in the chat box
# role="user" means this message is FROM the user (not the agent)
message = client.messages.create(
    thread_id=thread.id,                          # which thread to add this message to
    role="user",                                   # who sent it
    content="Show me all MBDs for the APAC region"  # the actual question
)
print("Message ID:", message.id)
print("✅ Thread created and message added")

In [ ]:
# ---- Cell 7: Start a Run and handle tool calls ----
# This is THE most important cell in the entire module.
# A Run = one execution of the agent loop on this thread.
# The loop: think → maybe call tool → observe result → think again → reply

import time   # time.sleep() lets us wait a moment between checking run status

# Start the run — this tells Foundry Agent Service:
# "take the thread, run the agent loop, produce a reply"
run = client.runs.create(
    thread_id=thread.id,    # which thread to run on
    agent_id=agent.id       # which agent to use
)
print("Run started. Run ID:", run.id)
print("Initial status:", run.status)

# The run happens ASYNCHRONOUSLY on Azure's servers
# We poll (check repeatedly) until it finishes or needs our help
# Possible statuses:
#   "queued"            → waiting to start
#   "in_progress"       → agent is thinking
#   "requires_action"   → agent wants to call a CUSTOM FUNCTION — needs us to run it
#   "completed"         → done, reply is ready
#   "failed"            → something went wrong

while run.status in ["queued", "in_progress", "requires_action"]:
    time.sleep(1)   # wait 1 second before checking again — don't hammer the API
    
    # Fetch the latest run status from Foundry
    run = client.runs.get(thread_id=thread.id, run_id=run.id)
    print("Status:", run.status)
    
    # THIS IS THE KEY MOMENT — agent wants to call one of our custom functions
    if run.status == "requires_action":
        
        # run.required_action holds the tool call(s) the agent wants to make
        # There can be multiple tool calls in one step
        tool_calls = run.required_action.submit_tool_outputs.tool_calls
        tool_outputs = []   # we'll collect our function results here
        
        for tool_call in tool_calls:
            fn_name = tool_call.function.name          # which function the agent chose
            fn_args = json.loads(tool_call.function.arguments)  # what args it decided to pass
            
            print(f"  → Agent calling: {fn_name}() with args: {fn_args}")
            
            # NOW WE RUN THE FUNCTION OURSELVES — in our environment, on our machine
            # The agent cannot run Python — it only knows the schema
            # WE execute the actual function and send the result back
            if fn_name == "get_mbds":
                result = get_mbds(**fn_args)        # ** unpacks the dict as keyword args
            elif fn_name == "calculate_cvs":
                result = calculate_cvs(**fn_args)   # same pattern
            else:
                result = json.dumps({"error": f"Unknown function: {fn_name}"})
            
            # Collect this result — we'll send ALL results back in one call
            tool_outputs.append({
                "tool_call_id": tool_call.id,   # must match the call ID so Foundry links result to call
                "output": result                 # the JSON string our function returned
            })
        
        # Send all tool results back to Foundry Agent Service
        # This resumes the agent loop — agent now reads our results and continues thinking
        client.runs.submit_tool_outputs(
            thread_id=thread.id,
            run_id=run.id,
            tool_outputs=tool_outputs    # list of {tool_call_id, output} dicts
        )
        print("  → Tool outputs submitted, agent resuming...")

print("✅ Run finished with status:", run.status)

In [ ]:
# ---- Cell 8: Read the agent's reply ----

# Now the run is complete — the agent's reply is stored as a Message in the Thread
# client.messages.list() fetches all messages in the thread
# They come back in reverse order (newest first), so the agent's reply is first

messages = client.messages.list(thread_id=thread.id)

# messages is a list of Message objects
# Each message has: role ("user" or "assistant"), content (list of content blocks)
for msg in messages:
    if msg.role == "assistant":    # "assistant" = the agent's reply
        # msg.content is a list — could be text, images, file references
        for block in msg.content:
            if hasattr(block, "text"):           # text block has a .text attribute
                print("Agent reply:")
                print(block.text.value)           # .value holds the actual string
        break   # stop after the first assistant message (the most recent reply)

In [ ]:
# ---- Cell 9: Send a follow-up message using SAME thread ----
# This proves thread memory — the agent remembers the APAC MBDs from Cell 6
# We don't create a new thread — we add to the existing one

message2 = client.messages.create(
    thread_id=thread.id,          # SAME thread ID — conversation continues
    role="user",
    content="Now calculate CVs for all three APAC MBDs"
    # Notice: we didn't say which MBDs — the agent knows from thread memory
)

# Start another run on the same thread
run2 = client.runs.create(
    thread_id=thread.id,
    agent_id=agent.id
)

# Same polling loop — paste exactly the same loop from Cell 7
while run2.status in ["queued", "in_progress", "requires_action"]:
    time.sleep(1)
    run2 = client.runs.get(thread_id=thread.id, run_id=run2.id)
    print("Status:", run2.status)
    
    if run2.status == "requires_action":
        tool_calls = run2.required_action.submit_tool_outputs.tool_calls
        tool_outputs = []
        
        for tool_call in tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"  → Agent calling: {fn_name}() with args: {fn_args}")
            
            if fn_name == "get_mbds":
                result = get_mbds(**fn_args)
            elif fn_name == "calculate_cvs":
                result = calculate_cvs(**fn_args)
            else:
                result = json.dumps({"error": f"Unknown function: {fn_name}"})
            
            tool_outputs.append({
                "tool_call_id": tool_call.id,
                "output": result
            })
        
        client.runs.submit_tool_outputs(
            thread_id=thread.id,
            run_id=run2.id,
            tool_outputs=tool_outputs
        )
        print("  → Tool outputs submitted...")

print("✅ Run 2 complete:", run2.status)

# Read reply — same pattern as Cell 8
messages2 = client.messages.list(thread_id=thread.id)
for msg in messages2:
    if msg.role == "assistant":
        for block in msg.content:
            if hasattr(block, "text"):
                print("\nAgent reply:")
                print(block.text.value)
        break

In [ ]:
# ---- Cell 10: Cleanup ----
# Always delete agents and threads you created during testing
# Agents cost nothing when idle but clutter your project
# Threads take up storage — clean them up when done

client.threads.delete(thread.id)    # delete the conversation
print("✅ Thread deleted:", thread.id)

client.delete_agent(agent.id)       # delete the agent definition
print("✅ Agent deleted:", agent.id)